# 187. EAGLE：特征级草稿与精确验证怎样实现？

> **面试问题：为什么 EAGLE 在 feature 而非 token 空间草稿，怎样用 target verifier、最长接受前缀和 branch KV 保证输出分布不变？**

## 先给结论

这题的关键不是调用框架，而是把状态、预算、验证器和可复放的制品合同显式化。教学代码用小数据验证不变量；生产实现仍需替换为真实模型、内核、隔离环境与线上观测。

## 一手资料

- [EAGLE](https://arxiv.org/abs/2401.15077)
- [Speculative Decoding](https://arxiv.org/abs/2211.17192)
- [EAGLE-3](https://arxiv.org/abs/2503.01840)


In [ ]:
import hashlib  # 导入本单元依赖。
import json  # 导入本单元依赖。
import math  # 导入本单元依赖。
from dataclasses import asdict, dataclass  # 导入本单元依赖。
VOCAB = ("A", "B", "C", "D")  # 计算并保存当前中间结果。
TARGET = ["A", "B", "D", "C"]  # 计算并保存当前中间结果。
assert len(VOCAB) == 4  # 用断言验证关键不变量。
assert len(TARGET) == 4  # 用断言验证关键不变量。
assert TARGET[0] in VOCAB  # 用断言验证关键不变量。


## 1. 最小状态与输入合同

先说明输入输出、边界和验证 oracle；再运行下面的底层实现。


In [ ]:
def longest_accepted(draft, target):  # 定义可复用的核心函数。
    accepted = 0  # 计算并保存当前中间结果。
    for proposal, truth in zip(draft, target):  # 遍历元素以累积状态。
        if proposal != truth:  # 按条件选择控制路径。
            break  # 计算并保存当前中间结果。
        accepted += 1  # 计算并保存当前中间结果。
    return accepted  # 返回当前计算结果。
draft = ["A", "B", "C"]  # 计算并保存当前中间结果。
assert longest_accepted(draft, TARGET) == 2  # 用断言验证关键不变量。
assert longest_accepted([], TARGET) == 0  # 用断言验证关键不变量。
assert longest_accepted(TARGET, TARGET) == len(TARGET)  # 用断言验证关键不变量。


## 2. 核心公式或状态转换

先说明输入输出、边界和验证 oracle；再运行下面的底层实现。


In [ ]:
def feature_draft(prefix, width):  # 定义可复用的核心函数。
    seed = sum(ord(token) for token in prefix)  # 计算并保存当前中间结果。
    return [VOCAB[(seed + step) % len(VOCAB)] for step in range(width)]  # 返回当前计算结果。
proposal = feature_draft(["A"], 3)  # 计算并保存当前中间结果。
assert len(proposal) == 3  # 用断言验证关键不变量。
assert set(proposal) <= set(VOCAB)  # 用断言验证关键不变量。
assert proposal == feature_draft(["A"], 3)  # 用断言验证关键不变量。


## 3. 候选选择与验证

先说明输入输出、边界和验证 oracle；再运行下面的底层实现。


In [ ]:
def verify_and_commit(prefix, draft, target_suffix):  # 定义可复用的核心函数。
    count = longest_accepted(draft, target_suffix)  # 计算并保存当前中间结果。
    committed = prefix + draft[:count]  # 计算并保存当前中间结果。
    if count < len(target_suffix):  # 按条件选择控制路径。
        committed.append(target_suffix[count])  # 计算并保存当前中间结果。
    return committed, count  # 返回当前计算结果。
committed, accepted = verify_and_commit(["A"], ["B", "C"], ["B", "D"])  # 计算并保存当前中间结果。
assert committed == ["A", "B", "D"]  # 用断言验证关键不变量。
assert accepted == 1  # 用断言验证关键不变量。
assert committed[-1] == "D"  # 用断言验证关键不变量。


## 4. 主路径实现

先说明输入输出、边界和验证 oracle；再运行下面的底层实现。


In [ ]:
@dataclass  # 计算并保存当前中间结果。
class BranchCache:  # 定义保存状态的数据结构。
    prefix: tuple  # 计算并保存当前中间结果。
    branches: dict  # 计算并保存当前中间结果。
    def fork(self, name, tokens):  # 定义可复用的核心函数。
        self.branches[name] = self.prefix + tuple(tokens)  # 计算并保存当前中间结果。
    def commit(self, name, count):  # 定义可复用的核心函数。
        return self.branches[name][:len(self.prefix) + count]  # 返回当前计算结果。
cache = BranchCache(("A",), {})  # 计算并保存当前中间结果。
cache.fork("draft", ["B", "C"])  # 计算并保存当前中间结果。
assert cache.commit("draft", 1) == ("A", "B")  # 用断言验证关键不变量。
assert cache.prefix == ("A",)  # 用断言验证关键不变量。
assert "draft" in cache.branches  # 用断言验证关键不变量。


## 5. 边界与失败分支

先说明输入输出、边界和验证 oracle；再运行下面的底层实现。


In [ ]:
def greedy_target(prefix, target):  # 定义可复用的核心函数。
    return prefix + target  # 返回当前计算结果。
exact = greedy_target(["A"], ["B", "D"])  # 计算并保存当前中间结果。
spec, _ = verify_and_commit(["A"], ["B", "C"], ["B", "D"])  # 计算并保存当前中间结果。
assert exact == spec  # 用断言验证关键不变量。
assert spec == ["A", "B", "D"]  # 用断言验证关键不变量。
assert len(spec) == 3  # 用断言验证关键不变量。


## 6. 正确性与基线对照

先说明输入输出、边界和验证 oracle；再运行下面的底层实现。


In [ ]:
def expected_cost(draft_width, accepted, target_cost=1.0, draft_cost=0.1):  # 定义可复用的核心函数。
    if draft_width <= 0 or accepted < 0 or accepted > draft_width:  # 按条件选择控制路径。
        raise ValueError("草稿宽度或接受数非法")  # 非法输入立即显式失败。
    return target_cost + draft_width * draft_cost, accepted + 1  # 返回当前计算结果。
cost, progress = expected_cost(3, 2)  # 计算并保存当前中间结果。
assert math.isclose(cost, 1.3)  # 用断言验证关键不变量。
assert progress == 3  # 用断言验证关键不变量。
assert expected_cost(1, 0)[1] == 1  # 用断言验证关键不变量。


## 7. 成本或数据合同

先说明输入输出、边界和验证 oracle；再运行下面的底层实现。


In [ ]:
@dataclass(frozen=True)  # 计算并保存当前中间结果。
class EAGLEArtifact:  # 定义保存状态的数据结构。
    target_version: str  # 计算并保存当前中间结果。
    draft_version: str  # 计算并保存当前中间结果。
    verify_mode: str  # 计算并保存当前中间结果。
artifact = EAGLEArtifact("target-v1", "feature-draft-v1", "exact-prefix")  # 计算并保存当前中间结果。
digest = hashlib.sha256(json.dumps(asdict(artifact), sort_keys=True).encode()).hexdigest()  # 计算并保存当前中间结果。
assert artifact.verify_mode == "exact-prefix"  # 用断言验证关键不变量。
assert len(digest) == 64  # 用断言验证关键不变量。
assert digest != hashlib.sha256(b"other").hexdigest()  # 用断言验证关键不变量。


## 8. 制品版本与面试收束

先说明输入输出、边界和验证 oracle；再运行下面的底层实现。


In [ ]:
assert longest_accepted(["A", "B"], ["A", "B"]) == 2  # 用断言验证关键不变量。
assert verify_and_commit([], ["A"], ["B"])[0] == ["B"]  # 用断言验证关键不变量。
assert cache.commit("draft", 2) == ("A", "B", "C")  # 用断言验证关键不变量。


## 面试收束

回答时按目标、状态合同、核心算法、失败分支、评测指标与发布版本组织；受控例子只证明实现不变量，不代表真实模型或生产系统性能。
